# Digitalify: from QONNX to bit-exact VHDL

This notebook walks through the full pipeline:

1. get a quantised network in **QONNX** format
2. import it into digitalify's integer IR
3. **verify** that the integer model matches the QONNX model
4. generate structural VHDL
5. **simulate** the VHDL with GHDL against the integer model

No PyTorch is needed: digitalify only depends on `numpy`, `onnx` and `qonnx`.

## 1. A QONNX model

In a real project the model comes from your training framework, for example Brevitas:

```python
from brevitas.export import export_qonnx
export_qonnx(brevitas_model, torch.randn(1, 16), "model.onnx")
```

or from QKeras via `qonnx.converters.from_keras`. Here we build one directly with
`digitalify.testing.make_mlp`, which emits the same structure Brevitas does:
`Quant → MatMul(Quant(W)) → Add(bias) → Relu → Quant → ...`

The network is a 16 → 8 → 4 MLP with **4-bit weights**, **6-bit unsigned hidden
activations**, INT8 input and INT8 output.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import onnx
import digitalify
from digitalify.testing import make_mlp

model = make_mlp([16, 8, 4], weight_bits=4, act_bits=6, output_bits=8, seed=1)
onnx.save(model, 'mlp_qonnx.onnx')      # digitalify accepts a path, ModelProto or ModelWrapper

for node in model.graph.node:
    print(f'{node.op_type:<8} {list(node.input)} -> {list(node.output)}')

Quant    ['global_in', 'inp_q_scale', 'inp_q_zeropt', 'inp_q_bitwidth'] -> ['inp_q']
Quant    ['W0', 'W0_q_scale', 'W0_q_zeropt', 'W0_q_bitwidth'] -> ['W0_q']
MatMul   ['inp_q', 'W0_q'] -> ['mm0']
Add      ['mm0', 'b0'] -> ['add0']
Relu     ['add0'] -> ['relu0']
Quant    ['relu0', 'act0_q_scale', 'act0_q_zeropt', 'act0_q_bitwidth'] -> ['act0_q']
Quant    ['W1', 'W1_q_scale', 'W1_q_zeropt', 'W1_q_bitwidth'] -> ['W1_q']
MatMul   ['act0_q', 'W1_q'] -> ['mm1']
Add      ['mm1', 'b1'] -> ['add1']
Quant    ['add1', 'global_out_scale', 'global_out_zeropt', 'global_out_bitwidth'] -> ['global_out']


## 2. Import into the integer IR

`from_qonnx` cleans the model up with `qonnx` (shape inference, constant folding,
batch size 1) and fuses each `MatMul → Add → Relu → Quant` chain into one `linear`
node with integer weights, an integer bias and a fixed-point requantisation
`q = clip(round(acc · MULT / 2^SHIFT))`.

In [2]:
ir = digitalify.from_qonnx('mlp_qonnx.onnx')
print(ir.summary())

layer = ir.nodes[0]
print()
print('weights (first row):', layer.attrs['weight'][0])
print('bias               :', layer.attrs['bias'])
print('requant MULT       :', layer.attrs['requant']['multiplier'])
print('requant SHIFT      :', layer.attrs['requant']['shift'])

IRGraph 'mlp': input (1, 1, 16) INT8 (scale 0.0625)
  linear_0       linear          (1, 1, 16) INT8 ->      (1, 1, 8) UINT6  W:INT4 acc:INT13 +relu requant(ROUND)
  linear_1       linear           (1, 1, 8) UINT6 ->      (1, 1, 4) INT8  W:INT3 acc:INT10 requant(ROUND)
output (1, 1, 4) INT8

weights (first row): [ 2  1 -1 -4  4 -1  2  0 -1  2  0  1  0  1  2 -3]
bias               : [ 27 -24  53   9 -47  60  48  35]
requant MULT       : [1 1 1 1 1 1 1 1]
requant SHIFT      : [6 6 6 6 6 6 6 6]


## 3. Verify against QONNX execution

`verify` feeds the same integer inputs to the bit-exact integer model and to
`qonnx.core.onnx_exec` running the original model, and compares the outputs in LSBs.
With power-of-two scales they must match exactly.

In [3]:
check = digitalify.verify('mlp_qonnx.onnx', n_vectors=256)
print(check.summary())

print()
print('input scale :', ir.input_scale)
print('output scale:', ir.output_scale)
for q_out in check.reference[3:6]:
    print('q_out =', q_out, '  real =', q_out * ir.output_scale)

[PASSED]  256/256 vectors match QONNX, max error 0 LSB

input scale : 0.0625
output scale: [0.015625 0.015625 0.015625 0.015625]
q_out = [-51  26  26  -2]   real = [-0.796875  0.40625   0.40625  -0.03125 ]
q_out = [-58  41  13  24]   real = [-0.90625   0.640625  0.203125  0.375   ]
q_out = [-41  28  15  12]   real = [-0.640625  0.4375    0.234375  0.1875  ]


## 4. Generate VHDL

One file per layer plus a structural top level. Entity names are prefixed with
the network name.

In [4]:
OUTPUT_DIR = 'generated_vhdl'
vhdl_files = digitalify.convert('mlp_qonnx.onnx', output_dir=OUTPUT_DIR)

for fname, src in vhdl_files.items():
    print(f'{fname:<20} {src.count(chr(10)):5d} lines')

mlp_skid.vhd            56 lines
mlp_linear_0.vhd       314 lines
mlp_linear_1.vhd       188 lines
mlp_top.vhd             98 lines


In [5]:
print(vhdl_files['mlp_top.vhd'])

-- Top-level structural entity for 'mlp'
-- Input : stream (1, 1, 16) (H, W, C) of INT8  (real = q * 0.0625), one pixel per beat
-- Output: stream (1, 1, 4) (H, W, C) of INT8
-- Handshake: a beat moves when valid and ready are both '1' (AXI4-Stream rules)
-- Stages:
--   linear_0       linear      fully parallel
--   linear_1       linear      fully parallel
-- Generated by digitalify

library ieee;
use ieee.std_logic_1164.all;
use ieee.numeric_std.all;

entity mlp_top is
    port (
        clk       : in  std_logic;
        rst       : in  std_logic;
        in_valid  : in  std_logic;
        in_ready  : out std_logic;
        in_data   : in  std_logic_vector(127 downto 0);
        out_valid : out std_logic;
        out_ready : in  std_logic;
        out_data  : out std_logic_vector(31 downto 0)
    );
end entity mlp_top;

architecture structural of mlp_top is

    -- Component declarations
    component mlp_linear_0 is
    port (
        clk       : in  std_logic;
        rst       :

In [6]:
# The linear component: constants, combinational MAC, registered requantisation
src = vhdl_files['mlp_linear_0.vhd']
start = src.index('    -- Combinational')
print(src[:1200])
print('    ...\n')
print(src[start:])

-- mlp_linear_0: linear  (1, 1, 16) x INT8 -> (1, 1, 8) x UINT6  (stream H, W, C)
-- fully parallel
-- QONNX node: MatMul_0
-- Generated by digitalify

library ieee;
use ieee.std_logic_1164.all;
use ieee.numeric_std.all;

entity mlp_linear_0 is
    port (
        clk       : in  std_logic;
        rst       : in  std_logic;
        in_valid  : in  std_logic;
        in_ready  : out std_logic;
        in_data   : in  std_logic_vector(127 downto 0);
        out_valid : out std_logic;
        out_ready : in  std_logic;
        out_data  : out std_logic_vector(47 downto 0)
    );
end entity mlp_linear_0;

architecture rtl of mlp_linear_0 is

    constant IN_F  : natural := 16;
    constant OUT_F : natural := 8;
    constant X_W   : natural := 8;    -- input element (INT8)
    constant W_W   : natural := 4;    -- weight (INT4)
    constant SUM_W : natural := 13;    -- adder tree
    constant ACC_W : natural := 13;    -- sum * 2^F + bias
    constant Y_W   : natural := 6;    -- output elemen

## 5. Error handling

Anything without a hardware mapping is rejected at import time, before any VHDL
is written: `UnsupportedOperatorError` for unknown operators or graph shapes,
`QONNXImportError` for quantisation schemes integer hardware cannot express.

In [7]:
from onnx import TensorProto, helper, numpy_helper
from digitalify import UnsupportedOperatorError, QONNXImportError

D = 'qonnx.custom_op.general'
inits = [numpy_helper.from_array(np.array(v, dtype=np.float32), n)
         for n, v in [('s', 0.25), ('z', 0.0), ('b', 8.0)]]
nodes = [
    helper.make_node('Quant', ['x', 's', 'z', 'b'], ['xq'], domain=D, signed=1, narrow=0),
    helper.make_node('Sigmoid', ['xq'], ['y']),
]
graph = helper.make_graph(nodes, 'unsupported',
    [helper.make_tensor_value_info('x', TensorProto.FLOAT, [1, 4])],
    [helper.make_tensor_value_info('y', TensorProto.FLOAT, [1, 4])], initializer=inits)
bad = helper.make_model(graph, opset_imports=[helper.make_opsetid('', 13), helper.make_opsetid(D, 1)])

try:
    digitalify.convert(bad)
except (UnsupportedOperatorError, QONNXImportError) as e:
    print(f'{type(e).__name__}: {e}')

UnsupportedOperatorError: Operator 'Sigmoid' is not supported (node 'Sigmoid_0'). The elementwise chain ['Sigmoid'] must end with a Quant node to be mapped (it becomes a lookup table).


## 6. Simulate with GHDL

`simulate` generates a self-checking testbench whose expected outputs come from
the integer model, and runs it through GHDL. Together with step 3 this shows
that the VHDL computes exactly what the QONNX model computes.

GHDL must be on `PATH` or pointed to by the `DIGITALIFY_GHDL` environment variable
(Windows: unzip a release from https://github.com/ghdl/ghdl/releases;
macOS: `brew install ghdl`; Ubuntu: `sudo apt install ghdl`).

In [8]:
import re
from digitalify.testing import find_ghdl

try:
    find_ghdl()
    result = digitalify.simulate('mlp_qonnx.onnx', n_vectors=16)
    print(result.summary())
    for line in result.log.splitlines():
        m = re.search(r'(Test \d+: \w+.*|TB SUMMARY.*)', line)
        if m:
            print('  ' + m.group(1))
except FileNotFoundError as e:
    print(e)

[PASSED]  16 passed, 0 failed, 41 cycles (first output after 8)
  Test 0: PASSED
  Test 1: PASSED
  Test 2: PASSED
  Test 3: PASSED
  Test 4: PASSED
  Test 5: PASSED
  Test 6: PASSED
  Test 7: PASSED
  Test 8: PASSED
  Test 9: PASSED
  Test 10: PASSED
  Test 11: PASSED
  Test 12: PASSED
  Test 13: PASSED
  Test 14: PASSED
  Test 15: PASSED
  TB SUMMARY: 16 passed, 0 failed out of 16 tests


## Next steps

* Synthesise the files in `generated_vhdl/` with Vivado — locally or through the
  `vivado_api` service in this repository (top entity: `mlp_top`).
* Use your own model: `digitalify.convert('model.onnx', output_dir=...)`.
* To support a new operator, map its QONNX pattern in
  `digitalify/frontend/qonnx_importer.py`, add a `VHDLOperator` in
  `digitalify/operators/`, register it, and add its bit-exact model to
  `digitalify/testing/reference.py`.